In [1]:
%load_ext autoreload
%autoreload 2

In [9]:
import torch
from torch.nn import functional as F
import math

In [35]:
from models.switch.switch_ff import SwitchFeedForward
from labml_nn.transformers.feed_forward import FeedForward
from transformers.activations import FastGELUActivation, GELUActivation

C:\Users\Wojtek\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [36]:
layer = SwitchFeedForward(
                is_scale_prob=True,
                n_experts=4,
                expert=FeedForward(
                    d_model=512,
                    d_ff=512*4,
                    dropout=0,
                    activation=GELUActivation(),
                ),
                d_model=512,
            )

In [37]:
x = torch.rand((16, 30, 512))
x = layer(x)

In [ ]:
# path = "datasets/ucf101/test_train_splits/ucfTrainTestlist/"
# file = "trainlist01.txt"
# with open(path+file, 'r') as file:
#     lines = [line.strip() for line in file.readlines()]
#     files = [line.split(" ")[0] for line in lines]
#     classes = [int(line.split(" ")[1]) for line in lines]
#     idxs = list(range(len(lines)))
#     train, test, _, _ = train_test_split(idxs, idxs, test_size=0.8, stratify=classes)
#     filtered_examples = []
#     for i in idxs:
#         if i in idxs:
#             filtered_examples.append(f"{files[i]} {classes[i]}")

In [ ]:
import wandb
from datasets import get_dataset
import torch
from utils import save_run, load_run
from weight_inits import init_timesformer
from torch.utils.data import DataLoader
from models.vivit_custom.utils.scheduler import WarmupCosineSchedule
from transforms import get_transforms, Div255

from constants import get_classes
from transformers import TimesformerConfig, TimesformerForVideoClassification
from transformers import ViTForImageClassification
from pytorchvideo.transforms import MixUp
import torch
from torchvision.transforms.v2 import (
    ScaleJitter,
    RandAugment,
    RandomCrop,
    Compose,
    ColorJitter,
    RandomHorizontalFlip,
    Normalize,
    RandomApply,
    UniformTemporalSubsample,
    CenterCrop,
    Lambda,
)
from pytorchvideo.transforms import UniformTemporalSubsample, ApplyTransformToKey
from loaders import get_loaders

In [ ]:
import random
import numpy as np

SEED = 0
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
config = {
    "base_learning_rate": 0.001,
    "label_smoothing": 0.2,
    "epochs": 10,
    "frames": 8,
    "val_epochs": 1,
    "batch_size": 20,
    "weight_decay": 1e-5,
    "load_run": False,
    "init_vit": True,
    "vit_base_size": "tiny",
    "cosine_schedule": True,
    "augmentations": True,
    "advanced_augmentations": True,
    "mixup": False,
    "mixup_alpha": 0.2,
    "dataset": "ucf101",
}

In [ ]:
CLASSES_LIST = get_classes(config["dataset"])

ds_transform_train, ds_transform_valid, ds_transform_test = get_transforms(config, CLASSES_LIST, config['dataset'])

dataset_train, dataset_valid, dataset_test, num_labels = get_dataset(
    config["dataset"], ds_transform_train, ds_transform_valid, ds_transform_test
)

training_loader, validation_loader, _ = get_loaders(
    dataset_train, dataset_valid, dataset_test, SEED, 5, n_workers=1
)
from utils import show_img_chw

In [ ]:
for i in validation_loader:
    show_img_chw(i['video'][0][0], 1)
    break

In [ ]:
dataset_valid.transform = transform

In [ ]:
show_img_chw(dataset_valid[0][0][0], denormalize=True)

In [ ]:
from vit_configs import get_vit_config
from transformers import TimesformerConfig, TimesformerForVideoClassification, VivitConfig, VivitForVideoClassification
from transformers import ViTForImageClassification
from weight_inits import init_timesformer


VIT_CONFIG = get_vit_config('tiny')
vit = ViTForImageClassification.from_pretrained(f"WinKawaks/vit-tiny-patch16-224")
model_config = VivitConfig(num_frames=config["frames"], **VIT_CONFIG, num_labels=51)
vivit = VivitForVideoClassification(model_config)


VIT_CONFIG = get_vit_config('tiny')
timesformer = ViTForImageClassification.from_pretrained(f"WinKawaks/vit-tiny-patch16-224")
model_config = TimesformerConfig(num_frames=config["frames"], **VIT_CONFIG, num_labels=51)
timesformer = TimesformerForVideoClassification(model_config)


In [ ]:
timesformer.timesformer.encoder.layer[0].temporal_dense.weight

In [ ]:
from torchvision.datasets import HMDB51, UCF101
from torch.utils.data import Subset
from pytorchvideo.data.hmdb51 import Hmdb51
from pytorchvideo.data.ucf101 import Ucf101
from pytorchvideo.data.clip_sampling import UniformClipSampler
import os
import pickle
from sklearn.model_selection import train_test_split
import numpy as np


CLASSES_LIST = get_classes("ucf101")
ds_transform_train, ds_transform_valid, test_transform = get_transforms(config, CLASSES_LIST, config["dataset"])

In [ ]:
import torch
from torchvision.transforms.v2 import (
    ScaleJitter,
    RandAugment,
    RandomCrop,
    Compose,
    ColorJitter,
    RandomHorizontalFlip,
    Normalize,
    RandomApply,
    UniformTemporalSubsample,
    CenterCrop,
    Lambda,
)
from pytorchvideo.transforms import UniformTemporalSubsample, ApplyTransformToKey
from utils import show_img_chw


In [ ]:
def swap_dims(x):
    return x.permute(1, 0, 2, 3)

def convert_dtype(x):
    return x.type(torch.uint8)


test_transforms_list = [
    ApplyTransformToKey(
        key="video",
        transform=Compose(
            [
                Lambda(convert_dtype),
                Lambda(swap_dims),
                # SplitEqually(num_frames=64),
                UniformTemporalSubsample(num_samples=8, temporal_dim=0),
                CenterCrop(size=(224, 224)),
                # Div255(),
                # Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ]
        ),
    )
]
test_transform = Compose(test_transforms_list)

dataset = Ucf101(
            data_path=r"datasets/ucf101/test_train_splits/ucfTrainTestlist/testlist01.txt",
            clip_sampler=UniformClipSampler(2),  # sample whole clips
            decode_audio=False,
            video_path_prefix="datasets/ucf101/data/UCF-101",
            transform=test_transform,
        )

In [ ]:
cnt = 0
import matplotlib.pyplot as plt

for i in dataset:
    # print(i['video'].shape)
    cnt += 1
    # show_img_chw(i['video'][0])
    # plt.show()
    if cnt > 5:
        break
    print(i['label'])

In [ ]:
loader = DataLoader(
    dataset=dataset,
    batch_size=12,
)

In [ ]:
for i in loader:
    print(i['video'].shape)
    print(i['label'])
    break

In [ ]:
dataset

In [ ]:
d2 = Subset(dataset, [0, 1])

In [ ]:
cnt = 0
import matplotlib.pyplot as plt

for i in d2:
    # print(i['video'].shape)
    cnt += 1
    # show_img_chw(i['video'][0])
    # plt.show()
    if cnt > 100:
        break
    print(i['label'])

In [2]:
import torch

In [3]:
a = torch.zeros([57, 3, 240, 320])

clip_frames = 64
frames = a.shape[0]
print(f"input: {a.shape}")

if frames < clip_frames:
    repeated = a[-1].unsqueeze(0).repeat((clip_frames - frames, 1, 1, 1))
    a = torch.cat((a, repeated), dim=0)

frames = a.shape[0]
mid_frame = frames // 2
left_idx_1 = max(0, mid_frame - clip_frames // 2 - clip_frames)
right_idx_1 = left_idx_1 + clip_frames
left_idx_2 = mid_frame - clip_frames // 2
right_idx_2 = mid_frame + clip_frames // 2
right_idx_3 = min(mid_frame + clip_frames // 2 + clip_frames, a.shape[0])
left_idx_3 = right_idx_3 - clip_frames
a = torch.stack((a[left_idx_1:right_idx_1], a[left_idx_2:right_idx_2], a[left_idx_3:right_idx_3]))

input: torch.Size([57, 3, 240, 320])
